## Datasci 530 - Project 1
### Product Scraper for St Andrews Links Online Shop

Joelle Millar
October 2026


Scraping occurred on Wednesday 21st October. The data collected is stored in `jlindeberg_products.csv` and details on the scraping process are stored in `collection_log.json`.

The navigation opens a chrome browser, navigates to `shop.standrews.com`, selects the `J. Lindeberg` brand, obtains details on each product stored for that brand and stores it in the files mentioned above.

The scraper collected 11 product observations from the J.Lindeberg section of the St Andrews Links online shop. Missing values were assessed for each variable, with the description and category fields checked for completeness. Product prices were converted to numeric values to calculate descriptive statistics. The mean price was £169, with a sample standard deviation of £49.

#### Set Up

Import packages from external python script and assign global variables.

In [424]:
# Import packages from external python script
exec(open("./scripts/import_packages.py").read())

In [ ]:
# Assign global variables
BASE_URL = 'https://shop.standrews.com'
STARTING_URL = BASE_URL + '/homepage/'

BRAND_NAME = 'J.Lindeberg'
MAX_PRODUCTS = 50           # restriction placed on collection
DELAY_SECONDS = 2

OUTPUT_CSV = 'jlindeberg_products.csv'
LOG_JSON = 'collection_log.json'

# XPath selectors used to locate website elements
BRAND_XPATH = (
    '//*[@id="root"]/div/div[1]/div[1]/div/div[2]/div[3]/nav/div/ul/li[6]/span'
)
BRAND_LINK_XPATH = f'//a[@title="{BRAND_NAME}"]'
PRODUCT_LINK_XPATH = '//a[contains(@href, "/catalog/products/")]'
COOKIE_XPATH = '//*[@id="root"]/div/div[4]/div[2]/button[2]'
PRODUCT_CARD_XPATH = (
    '//*[@id="root"]/div/div[2]/div[2]/div[3]/div[2]/div[2]/div[2]/ul/li'
)
CARD_NAME_XPATH = './div/a/div[2]/div/h3'
CARD_PRICE_XPATH = './div/a/div[2]/div/div[3]/span/span'

DESCRIPTION_XPATH = '//*[@id="root"]/div/div[2]/div[2]/div[2]/div/div/div[3]/div[4]/ul/li[1]'

#### Helper Functions

Check website permissions, start chrome browser, accept cookies banner, open brands category from navigation bar, select J.Lindeberg, collect product information and extract details

In [426]:
# Check website permissions
robots = robotparser.RobotFileParser()
robots.set_url(f"{BASE_URL}/robots.txt")
robots.read()

def check_robots(url):
    """Return whether robots.txt permits access to a url."""
    return robots.can_fetch("*", url)

In [427]:
# Start Chrome Browser
def make_driver():
    return webdriver.Chrome()

In [428]:
# Accept the cookie banner if it appears
def accept_cookies(driver):
    """Click 'Accept necessary only' if the banner appears."""
    try:
        button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, COOKIE_XPATH))
        )
        button.click()
        print("Accepted necessary cookies only")
    except TimeoutException:
        print("Cookie button not found, continuing")

In [429]:
# Open the brands menu from the Navigation bar and select J.Lindeberg
def open_brand(driver):
    wait = WebDriverWait(driver, 10)
    
    # Open the Brands dropdown 
    wait.until(
        EC.element_to_be_clickable((By.XPATH, BRAND_XPATH))
    ).click()

    # Find the link for the selected brand
    brand_xpath = f"//a[@title='{BRAND_NAME}']"
    
    brand_link = wait.until(
        EC.presence_of_element_located((By.XPATH, brand_xpath))
    )

    # Javascript click handles the site's overlapping navigation elements
    driver.execute_script("arguments[0].click();", brand_link)

    # Wait for the brand's product links to load
    wait.until(
        EC.presence_of_element_located((By.XPATH, PRODUCT_LINK_XPATH))
    )

    print("Brand page: ", driver.current_url)

In [430]:
# Collect product names, prices, and URLs from the listing page
def collect_product_links(driver):
    wait = WebDriverWait(driver, 10)

    wait.until(
        EC.presence_of_element_located((By.XPATH, PRODUCT_LINK_XPATH))
    )

    products = []
    seen_urls = set()

    for card in driver.find_elements(By.XPATH, PRODUCT_CARD_XPATH):
        try:
            link = card.find_element(
                By.XPATH, ".//a[contains(@href, '/catalog/products/')]"
            )
            url = link.get_attribute("href")

            # Skip duplicate URLs
            if not url or url in seen_urls:
                continue

            # Extract the product name from this card
            name = card.find_element(
                By.XPATH,
                CARD_NAME_XPATH
            ).text.strip()

            # Use the price location from the XPath, relative to this specific card
            price = card.find_element(
                By.XPATH,
                CARD_PRICE_XPATH
            ).text.strip()

            products.append({
                "name": name,
                "price": price,
                "url": url,
            })
            seen_urls.add(url)

        except NoSuchElementException:
            print("Skipped a product card with missing information")

        if len(products) >= MAX_PRODUCTS:
            break

    return products


In [431]:
# Visit a product page and extract its details
def scrape_product(driver, url):
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    # Wait for the product title to load
    name = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//h1"))
    ).text.strip()

    # Click the Description block to expand it
    description_block = wait.until(
        EC.element_to_be_clickable((By.XPATH, DESCRIPTION_XPATH))
    )

    driver.execute_script(
        "arguments[0].click();", description_block
    )

    # Extract the description text after expanding
    description = wait.until(
        EC.visibility_of_element_located((By.XPATH, DESCRIPTION_XPATH))
    ).text.strip()

    # Extract breadcrumb links for product categories
    breadcrumbs = driver.find_elements(
        By.XPATH,
        "//*[contains(@class, 'readcrumb')]//a "
        "| //nav[@aria-label='breadcrumb']//a"
    )

    categories = " > ".join(
        item.text.strip()
        for item in breadcrumbs
        if item.text.strip()
    ) or None

    return {
        "name": name,
        "description": description or None,
        "categories": categories,
        "url": url,
    }

### Compliance Check

In [432]:
product_path = f"{BASE_URL}/catalog/products/"

robot_checks = {
    "starting_url": check_robots(STARTING_URL),
    "product_path": check_robots(product_path)
}

if not all(robot_checks.values()):
    raise SystemExit("robots.txt does not permit one or more target paths")

print("robots.txt permits the target paths")

robots.txt permits the target paths


### Start Browser, open homepage, accept cookies

In [433]:
# Record the start time and initialise the output lists
started = datetime.now().isoformat(timespec="seconds")
records = []
errors = []

driver = make_driver()

try:
    # Open the website      
    driver.get(STARTING_URL)
    print(driver.title, "|", driver.current_url)

    # Dismiss the cookie banner and navigate to the selected brand
    accept_cookies(driver)
    open_brand(driver)

    # Collect product URLs
    products = collect_product_links(driver)
    print("Products found:", len(products))

    # Visit each product page and save its extracted information
    for i, product in enumerate(products, start=1):
        try:
            details = scrape_product(driver, product["url"])
            
            record = {
                "name": product["name"],
                "vendor": BRAND_NAME,
                "price": product["price"],
                "description": details["description"],
                "categories": details["categories"],
                "url": product["url"],
                "scraped_at": datetime.now().isoformat(
                    timespec="seconds"
                ),
            }

            records.append(record)

            print(
                f"[{i}/{len(products)}] "
                f"{record['name']} | {record['price']}"
            )

        except Exception as error:
            errors.append({
                "url": product.get("url"),
                "error": type(error).__name__,
                "message": str(error),
            })

            print(f"Failed to scrape: {product['url']}")
            
        # Pause between product-page requests
        if i < len(products):
            time.sleep(DELAY_SECONDS)
            
finally:
    # Close Chrome even if the scraper encounters an error
    driver.quit()


St Andrews Links : The Home of Golf | https://shop.standrews.com/homepage/
Accepted necessary cookies only
Brand page:  https://shop.standrews.com/catalog/category/jlindeberg
Products found: 11
[1/11] J.lindeberg Heyden Quilt Hybrid Jacket | £240.00
[2/11] J.lindeberg Heath Polo | £120.00
[3/11] J.lindeberg Tour Tech Polo | £120.00
[4/11] J.lindeberg Tour Tech Polo | £125.00
[5/11] J.lindeberg Tour Tech Print Polo | £135.00
[6/11] J.lindeberg Lukas 1/4 Zip Midlayer | £185.00
[7/11] J.lindeberg Club Hoodie | £180.00
[8/11] J.lindeberg Heath Polo | £110.00
[9/11] J.lindeberg Heyden Quilt Hybrid Jacket | £240.00
[10/11] J.lindeberg Heyden Quilt Hybrid Vest | £205.00
[11/11] J.lindeberg Sole Hoodie | £200.00


### Close browser, save data and log products

In [434]:
# Convert the collected records into a DataFrame
df = pd.DataFrame(records)

# Create a copy for analysis and create a table with descriptive statistics
analysis_df = df.copy()

# Convert price from text to numeric
analysis_df["price"] = pd.to_numeric(
    analysis_df["price"]
    .astype("string")
    .str.replace(r"[£,\s]", "", regex=True),
    errors="coerce"
)

# Table: Number of observations and missing values
missing_table = pd.DataFrame({
    "Variable": analysis_df.columns,
    "Observations": analysis_df.notna().sum().values,
    "Missing values": analysis_df.isna().sum().values
})

print("Number of product records:", len(analysis_df))

# Display formatted table
display(
    missing_table.style
    .hide(axis="index")
    .set_caption("Data completeness by variable")
    .format({
        "Non-missing": "{:,.0f}",
        "Missing": "{:,.0f}"
    })
    .set_properties(**{
        "text-align": "left",
        "padding": "8px 14px"
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "8px 0")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("padding", "8px 14px"),
                ("background-color", "#eee8f0"),
                ("color", "#302637")
            ]
        },
        {
            "selector": "td",
            "props": [("border-bottom", "1px solid #e5e5e5")]
        }
    ])
)

# Table: Mean and variance of numeric variables
numeric_summary = (
    analysis_df[["price"]]
    .select_dtypes(include="number")
    .agg(["count", "mean", "std", "min", "median", "max"])
    .T
    .rename(columns={
        "count": "N",
        "mean": "Mean (£)",
        "std": "Std. deviation (£)",
        "min": "Minimum (£)",
        "median": "Median (£)",
        "max": "Maximum (£)"
    })
    .reset_index(names="Variable")
)

# Display formatted table
display(
    numeric_summary.style
    .hide(axis="index")
    .set_caption("Descriptive statistics for product prices")
    .format({
        "N": "{:,.0f}",
        "Mean (£)": "£{:,.2f}",
        "Std. deviation (£)": "£{:,.2f}",
        "Minimum (£)": "£{:,.2f}",
        "Median (£)": "£{:,.2f}",
        "Maximum (£)": "£{:,.2f}"
    }, na_rep="—")
    .set_properties(**{
        "text-align": "right",
        "padding": "8px 14px"
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "8px 0")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "right"),
                ("padding", "8px 14px"),
                ("background-color", "#eee8f0"),
                ("color", "#302637")
            ]
        },
        {
            "selector": "td",
            "props": [("border-bottom", "1px solid #e5e5e5")]
        }
    ])
)

# Save the product data as a CSV file
df.to_csv(OUTPUT_CSV, index=False)

# Record how the collection was conducted
log = {
    "website": BASE_URL,
    "brand": BRAND_NAME,
    "collection_started": started,
    "collection_finished": datetime.now().isoformat(timespec="seconds"),
    "observations_collected": len(df),
    "pages_skipped": len(errors),
    "restrictions": {
        "max_products": MAX_PRODUCTS,
        "delay_seconds_between_requests": DELAY_SECONDS,
    },
    "robots_check": robot_checks,
    "errors": errors,
}

with open(LOG_JSON, "w") as file:
    json.dump(log, file, indent=2)

print(f"Saved {len(df)} products to {OUTPUT_CSV}")
print(f"Saved collection log to {LOG_JSON}")

Number of product records: 11


Variable,Observations,Missing values
name,11,0
vendor,11,0
price,11,0
description,11,0
categories,11,0
url,11,0
scraped_at,11,0


Variable,N,Mean (£),Std. deviation (£),Minimum (£),Median (£),Maximum (£)
price,11,£169.09,£49.08,£110.00,£180.00,£240.00


Saved 11 products to jlindeberg_products.csv
Saved collection log to collection_log.json
